In [9]:
import pandas as pd
import os


In [ ]:


# ─────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────

CSV_FILE   = "1_run_mapper_eval_full_data/1_run_mapper_eval_full_data.csv"
OUTPUT_DIR = "latex_output"

DATASET_LABELS = {
    "restaurant": "Restaurant",
    "tax":        "Tax",
    "sisu":       "SISU",
}

VARIANT_LABELS = {
    "LLM GPT-5-mini (with_examples_and_table)":   "LLM (GPT-5-mini)",
    "MultiStep GPT-5-mini (less_restrictive)":     "MultiStep (GPT-5-mini)",
    "LLM Mistral-small (with_examples_and_table)": "LLM (Mistral-small)",
    "MultiStep Mistral-small(less_restrictive)":   "MultiStep (Mistral-small)",
}

# ── Two blocks separated by a visible gap in the column spec ──
#
#   LEFT  block — evaluated over ALL sentences
#   RIGHT block — evaluated over GT constraint sentences only

LEFT_GROUPS = [
    ("Rule Detection",            ["clf_precision",                "clf_recall",                "clf_f1"]),
    ("Viol.\\ Discovery (Aggr.)", ["aggregate_violation_precision","aggregate_violation_recall","aggregate_violation_f1"]),
]

RIGHT_GROUPS = [
    ("Constraint Extraction",     ["constraint_exact_precision",   "constraint_exact_recall",   "constraint_exact_f1"]),
    ("Viol.\\ Discovery (Pair.)", ["pairwise_violation_precision", "pairwise_violation_recall", "pairwise_violation_f1"]),
]

SUBHEADERS = ["P", "R", "F$_1$"]

# Bold the best value per dataset block for these columns
BOLD_MAX_COLS = [
    "clf_precision", "clf_recall", "clf_f1",
    "aggregate_violation_precision", "aggregate_violation_recall", "aggregate_violation_f1",
    "constraint_exact_precision", "constraint_exact_recall", "constraint_exact_f1",
    "pairwise_violation_precision", "pairwise_violation_recall", "pairwise_violation_f1",
]



In [11]:
# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def fmt(val, bold=False):
    try:
        s = f"{float(val):.2f}"
    except (ValueError, TypeError):
        s = "--"
    return f"\\textbf{{{s}}}" if bold else s


def _all_cols(groups):
    return [col for _, cols in groups for col in cols]


# ─────────────────────────────────────────────────────────────
# Table generator
# ─────────────────────────────────────────────────────────────

def generate_table(df: pd.DataFrame) -> str:

    left_cols  = _all_cols(LEFT_GROUPS)
    right_cols = _all_cols(RIGHT_GROUPS)
    all_cols   = left_cols + right_cols

    n_left  = len(left_cols)
    n_right = len(right_cols)
    n_total = 1 + n_left + n_right

    # ── column spec ──────────────────────────────────────────
    # l  [left groups]  [big gap]  [right groups]
    left_spec  = "".join(["@{\\hspace{0.7em}}rrr" for _ in LEFT_GROUPS])
    right_spec = "".join(["@{\\hspace{0.7em}}rrr" for _ in RIGHT_GROUPS])
    col_spec   = "l" + left_spec + "@{\\hspace{1.6em}}" + right_spec

    lines = []
    lines.append(r"\begin{table*}[t]")
    lines.append(r"  \centering")
    lines.append(r"  \small")
    lines.append(r"  \setlength{\tabcolsep}{3.5pt}")
    lines.append(f"  \\begin{{tabular}}{{{col_spec}}}")
    lines.append(r"    \toprule")

    # ── ROW 1: block scope labels ─────────────────────────────
    left_start  = 2
    left_end    = 1 + n_left
    right_start = left_end + 1
    right_end   = left_end + n_right

    lines.append(
        f"    & \\multicolumn{{{n_left}}}{{c}}{{\\textit{{All sentences}}}} & "
        f"\\multicolumn{{{n_right}}}{{c}}{{\\textit{{GT sentences only}}}} \\\\"
    )
    lines.append(
        f"    \\cmidrule(lr){{{left_start}-{left_end}}}"
        f"\\cmidrule(lr){{{right_start}-{right_end}}}"
    )

    # ── ROW 2: group names ────────────────────────────────────
    group_headers = []
    cmidrules     = []
    col_cursor    = 2
    for group_name, cols in LEFT_GROUPS + RIGHT_GROUPS:
        span = len(cols)
        group_headers.append(f"\\multicolumn{{{span}}}{{c}}{{{group_name}}}")
        cmidrules.append(f"\\cmidrule(lr){{{col_cursor}-{col_cursor + span - 1}}}")
        col_cursor += span

    lines.append(f"    \\textbf{{Method}} & {' & '.join(group_headers)} \\\\")
    lines.append(f"    {'  '.join(cmidrules)}")

    # ── ROW 3: P / R / F1 sub-headers ────────────────────────
    subheader_cells = []
    for _, cols in LEFT_GROUPS + RIGHT_GROUPS:
        for sub in SUBHEADERS:
            subheader_cells.append(sub)
    lines.append(f"    & {' & '.join(subheader_cells)} \\\\")
    lines.append(r"    \midrule")

    # ── Data rows grouped by dataset ─────────────────────────
    datasets = df["dataset"].unique()

    for d_idx, dataset in enumerate(datasets):
        ddf = df[df["dataset"] == dataset].copy()
        dataset_label = DATASET_LABELS.get(dataset, dataset.capitalize())

        # Per-column max for bolding
        col_max = {}
        for col in BOLD_MAX_COLS:
            if col in ddf.columns:
                try:
                    col_max[col] = pd.to_numeric(ddf[col], errors="coerce").max()
                except Exception:
                    pass

        lines.append(
            f"    \\multicolumn{{{n_total}}}{{l}}"
            f"{{\\textit{{{dataset_label}}}}} \\\\"
        )

        for _, row in ddf.iterrows():
            variant = VARIANT_LABELS.get(row["variant_name"], row["variant_name"])
            cells = [f"~~{variant}"]

            for col in all_cols:
                val     = row.get(col, float("nan"))
                is_bold = (
                    col in col_max
                    and pd.notna(col_max[col])
                    and pd.notna(val)
                    and abs(float(val) - col_max[col]) < 1e-9
                )
                cells.append(fmt(val, bold=is_bold))

            lines.append(f"    {' & '.join(cells)} \\\\")

        if d_idx < len(datasets) - 1:
            lines.append(r"    \midrule")

    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")

    # ── Caption ───────────────────────────────────────────────
    caption = (
        "  \\caption{%\n"
        "    Results of the constraint extraction pipeline evaluated on two regulatory datasets.\n"
        "    The table is divided into two evaluation scopes.\n"
        "    \\textit{All sentences} (left): metrics computed over every sentence in the document.\n"
        "    \\textbf{Rule Detection} treats the mapper as an implicit classifier---a sentence\n"
        "    is predicted positive if the mapper produces any constraint for it.\n"
        "    \\textbf{Violation Discovery (Aggr.)} pools all predicted and all gold constraints\n"
        "    and measures row-level violation overlap at the dataset level (precision, recall, F\\textsubscript{1}).\n"
        "    \\textit{GT sentences only} (right): metrics restricted to sentences\n"
        "    that carry a ground-truth constraint.\n"
        "    \\textbf{Constraint Extraction} measures exact set-membership of predicted\n"
        "    vs.\\ gold predicate sets after normalisation.\n"
        "    \\textbf{Violation Discovery (Pair.)} measures, per GT sentence, whether\n"
        "    the predicted constraint identifies the same data violations as the gold constraint\n"
        "    (average precision, recall, F\\textsubscript{1} across sentences).\n"
        "    Best result per metric group and dataset in \\textbf{bold}.%\n"
        "  }"
    )
    lines.append(caption)
    lines.append(r"  \label{tab:main_results}")
    lines.append(r"\end{table*}")

    return "\n".join(lines)




In [12]:

df = pd.read_csv(CSV_FILE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = df[df["status"] == "success"].copy()
print(f"Loaded {len(df)} rows from {CSV_FILE}")
print(f"Datasets: {df['dataset'].unique().tolist()}")
print(f"Variants: {df['variant_name'].unique().tolist()}\n")

table = generate_table(df)



snippet_path = os.path.join(OUTPUT_DIR, "run_mapper_result.tex")
with open(snippet_path, "w") as f:
    f.write("% Include in your paper with \\input{run_mapper_result}\n")
    f.write(table)
print(f"Paper snippet:       {snippet_path}")

print("\n── Table preview ───────────────────────────────────\n")
print(table)

Loaded 2 rows from 1_run_mapper_eval_full_data/1_run_mapper_eval_full_data.csv
Datasets: ['person']
Variants: ['LLM GPT-5-mini (with_examples_and_table)', 'MultiStep GPT-5-mini (less_restrictive)']

Paper snippet:       latex_output/run_mapper_result.tex

── Table preview ───────────────────────────────────

\begin{table*}[t]
  \centering
  \small
  \setlength{\tabcolsep}{3.5pt}
  \begin{tabular}{l@{\hspace{0.7em}}rrr@{\hspace{0.7em}}rrr@{\hspace{1.6em}}@{\hspace{0.7em}}rrr@{\hspace{0.7em}}rrr}
    \toprule
    & \multicolumn{6}{c}{\textit{All sentences}} & \multicolumn{6}{c}{\textit{GT sentences only}} \\
    \cmidrule(lr){2-7}\cmidrule(lr){8-13}
    \textbf{Method} & \multicolumn{3}{c}{Rule Detection} & \multicolumn{3}{c}{Viol.\ Discovery (Aggr.)} & \multicolumn{3}{c}{Constraint Extraction} & \multicolumn{3}{c}{Viol.\ Discovery (Pair.)} \\
    \cmidrule(lr){2-4}  \cmidrule(lr){5-7}  \cmidrule(lr){8-10}  \cmidrule(lr){11-13}
    & P & R & F$_1$ & P & R & F$_1$ & P & R & F$_1$ & P & R 